In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib
import os

In [12]:
# %%
# -----------------------------------------------------------
# 1. Load trained anomaly models (NO retraining)
# -----------------------------------------------------------
try:
    dbscan_model = joblib.load("./builds/dbscan_model.pkl")
except Exception:
    dbscan_model = None

try:
    iso_model = joblib.load("./builds/isolation_forest_model.pkl")
except Exception:
    iso_model = None

print("Loaded models:")
print(f"  dbscan : {dbscan_model is not None}")
print(f"  iso    : {iso_model is not None}")

if iso_model is None and dbscan_model is None:
    raise RuntimeError("No trained IsolationForest or DBSCAN model found.")

Loaded models:
  dbscan : True
  iso    : True


In [13]:
# -----------------------------------------------------------
# 2. Load data and engineer features (same as other scripts)
# -----------------------------------------------------------
df = pd.read_csv("../../data/all_data.csv")

df["timestamp_parsed"] = pd.to_datetime(df["timestamp_parsed"], errors="coerce")
df["last_seen"] = pd.to_datetime(df["last_seen"], errors="coerce")

df["hour"] = df["timestamp_parsed"].dt.hour
df["day"] = df["timestamp_parsed"].dt.day
df["weekday"] = df["timestamp_parsed"].dt.weekday
df["time_alive"] = (df["last_seen"] - df["timestamp_parsed"]).dt.total_seconds()

In [14]:
# -----------------------------------------------------------
# 3. Select numeric features
# -----------------------------------------------------------
X_num = df.select_dtypes(include=[np.number])

In [15]:
# -----------------------------------------------------------
# 4. Impute + scale (fresh here; we are NOT loading old scalers)
# -----------------------------------------------------------
imputer = SimpleImputer(strategy="median")
X_cleaned = imputer.fit_transform(X_num)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cleaned)

In [16]:
# -----------------------------------------------------------
# 5. Build pseudo-labels from the loaded anomaly model
#    (no retraining, just predict)
# -----------------------------------------------------------
# Option A: use IsolationForest as teacher if available
if iso_model is not None:
    iso_preds = iso_model.predict(X_scaled)       # -1 = anomaly, 1 = normal
    y_pseudo = np.where(iso_preds == -1, 1, 0)    # 1 = anomaly, 0 = normal
    teacher_name = "IsolationForest"

# Option B: fallback to DBSCAN if IsolationForest not available
elif dbscan_model is not None:
    # DBSCAN likely trained on PCA-transformed data
    pca = joblib.load("./builds/pca_model.pkl")
    X_pca = pca.transform(X_scaled)
    dbscan_labels = dbscan_model.fit_predict(X_pca)  # DBSCAN has no predict
    y_pseudo = np.where(dbscan_labels == -1, 1, 0)   # 1 = noise/anomaly
    teacher_name = "DBSCAN"

print(f"\nUsing {teacher_name} outputs as pseudo-labels for AdaBoost.")
n_anom = np.sum(y_pseudo == 1)
print(f"Anomalies (1): {n_anom} / {len(y_pseudo)} ({n_anom/len(y_pseudo)*100:.2f}%)")


Using IsolationForest outputs as pseudo-labels for AdaBoost.
Anomalies (1): 35 / 3626 (0.97%)


In [17]:
# -----------------------------------------------------------
# 6. Train/Test split for AdaBoost (supervised on pseudo-labels)
# -----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_pseudo,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y_pseudo,
)

print("\nAdaBoost data split:")
print("  Train X:", X_train.shape, "Train y:", y_train.shape)
print("  Test  X:", X_test.shape, "Test  y:", y_test.shape)


AdaBoost data split:
  Train X: (2900, 12) Train y: (2900,)
  Test  X: (726, 12) Test  y: (726,)


In [18]:
# -----------------------------------------------------------
# 7. Train AdaBoost ONLY (base models are already trained)
# -----------------------------------------------------------
ada = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=1.0,
    random_state=42,
)

ada.fit(X_train, y_train)

AdaBoostClassifier(n_estimators=100, random_state=42)

In [19]:
# -----------------------------------------------------------
# 8. Simple performance report (accuracy, precision, recall, F1)
# -----------------------------------------------------------
y_pred = ada.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="binary", zero_division=0)
recall = recall_score(y_test, y_pred, average="binary", zero_division=0)
f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

print(f"\n=== AdaBoost TEST SET REPORT (vs {teacher_name} pseudo-labels) ===")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification report:\n")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))


=== AdaBoost TEST SET REPORT (vs IsolationForest pseudo-labels) ===
Accuracy : 0.9986
Precision: 1.0000
Recall   : 0.8571
F1-score : 0.9231

Classification report:

              precision    recall  f1-score   support

           0     0.9986    1.0000    0.9993       719
           1     1.0000    0.8571    0.9231         7

    accuracy                         0.9986       726
   macro avg     0.9993    0.9286    0.9612       726
weighted avg     0.9986    0.9986    0.9986       726



In [20]:
# -----------------------------------------------------------
# 9. Save AdaBoost model + local preprocessors (for reuse)
# -----------------------------------------------------------
os.makedirs("./builds", exist_ok=True)
joblib.dump(ada, "./builds/adaboost_model.pkl")
joblib.dump(imputer, "./builds/imputer.pkl")
joblib.dump(scaler, "./builds/scaler.pkl")
  
print("\n✅ AdaBoost model saved in ./builds/adaboost_model.pkl")
print(f"   Trained to mimic: {teacher_name}")



✅ AdaBoost model saved in ./builds/adaboost_model.pkl
   Trained to mimic: IsolationForest
